In [ ]:
!pip install -q -U faiss-cpu sentence-transformers

In [ ]:
import numpy as np
import pandas as pd
import torch
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OPTIONS = ["A", "B", "C", "D", "E"]

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
train = pd.read_csv(TRAIN_PATH)

In [ ]:
print("Creating knowledge base")
kb = [str(row[row["answer"]]) for _, row in train.iterrows()]

print("Loading embedding model and creating index")
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
kb_embeddings = embedder.encode(kb, show_progress_bar=True)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)
print("Knowledge base successfully created:", index.ntotal, "documents")

zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if DEVICE == "cuda" else -1,
)
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def retrieve(prompt: str, k: int):
    query_embedding = embedder.encode([prompt])
    distances, indices = index.search(query_embedding, k)
    return indices[0], distances[0]

## Q1

In [ ]:
row_150 = train.iloc[150]
prompt_150 = str(row_150["prompt"])
labels_150 = [str(row_150[opt]) for opt in OPTIONS]
ans_150 = str(row_150[row_150["answer"]])

result_150 = zs(prompt_150, candidate_labels=labels_150)
correct_prob_150 = result_150["scores"][result_150["labels"].index(ans_150)]
print("P(correct option) =", round(correct_prob_150, 3))

## Q2

In [ ]:
retrieved_indices, _ = retrieve(prompt_150, k=10)
rank_150 = list(retrieved_indices).index(150) + 1 if 150 in retrieved_indices else None

print("FAISS retrieval order (KB indices):", retrieved_indices)
print("Rank of the true document (KB index 150):", rank_150)

## Q3

In [ ]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)

docs_10 = [kb[i] for i in retrieved_indices]
pairs_150 = [[prompt_150, doc] for doc in docs_10]
ce_scores_150 = cross_encoder.predict(pairs_150)

reranked_order = [retrieved_indices[i] for i in np.argsort(ce_scores_150)[::-1]]
ce_rank_150 = reranked_order.index(150) + 1 if 150 in reranked_order else None

print("Cross-encoder reranked order:", reranked_order)
print("Rank of the true document after reranking:", ce_rank_150)

## Q4

In [ ]:
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])
top5_idx_42, _ = retrieve(prompt_42, k=5)
docs_5 = [kb[i] for i in top5_idx_42]

rag_string_42 = f"Context: {' '.join(docs_5)} Question: {prompt_42}"
n_tokens_42 = len(bert_tokenizer(rag_string_42, truncation=False)["input_ids"])
print("Total tokens:", n_tokens_42)

## Q5

In [ ]:
true_doc_150 = kb[150]
rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"

result_150_rag = zs(rag_string_150, candidate_labels=labels_150)
correct_prob_150_rag = result_150_rag["scores"][result_150_rag["labels"].index(ans_150)]
print("P(correct option | true context) =", round(correct_prob_150_rag, 3))

## Q6

In [ ]:
adversarial_doc = kb[999]
adversarial_string_150 = f"Context: {adversarial_doc} Question: {prompt_150}"

result_150_adv = zs(adversarial_string_150, candidate_labels=labels_150)
correct_prob_150_adv = result_150_adv["scores"][result_150_adv["labels"].index(ans_150)]
print("P(correct option | wrong context) =", round(correct_prob_150_adv, 3))

## Q7

In [ ]:
hits = 0
for i in range(100):
    row = train.iloc[i]
    prompt_i = str(row["prompt"])
    correct_text = str(row[row["answer"]])

    top5_idx, _ = retrieve(prompt_i, k=5)
    docs = [kb[j] for j in top5_idx]
    if any(correct_text in doc for doc in docs):
        hits += 1

hit_rate = round(hits / 100 * 100, 1)
print(f"Hit rate over first 100 rows: {hit_rate}%")

## Q8

In [ ]:
def calculate_map3(truth, top3):
    if truth == top3[0]:
        return 1.0
    if truth == top3[1]:
        return 0.5
    if truth == top3[2]:
        return 1 / 3
    return 0.0


rag_pipeline_map3_scores = []
for i in range(20):
    row = train.iloc[i]
    prompt_i = str(row["prompt"])
    labels_i = [str(row[opt]) for opt in OPTIONS]

    top5_idx, _ = retrieve(prompt_i, k=5)
    docs_i = [kb[j] for j in top5_idx]
    pairs_i = [[prompt_i, doc] for doc in docs_i]
    ce_scores_i = cross_encoder.predict(pairs_i)
    best_doc = docs_i[int(np.argmax(ce_scores_i))]

    rag_string_i = f"Context: {best_doc} Question: {prompt_i}"
    result_i = zs(rag_string_i, candidate_labels=labels_i)
    ranked_options_i = [OPTIONS[labels_i.index(lbl)] for lbl in result_i["labels"]]

    rag_pipeline_map3_scores.append(calculate_map3(row["answer"], ranked_options_i[:3]))

print("RAG pipeline MAP@3 (first 20 rows):", round(float(np.mean(rag_pipeline_map3_scores)), 3))